In [4]:
from src.models.autoencoder import AutoEncoder, AutoEncoderParams
from src.models.transformer import Flux2, Klein4BParams
from src.utils import checkpoint_transfer_diffusers2vae, checkpoint_transfer_diffusers2flux
from safetensors.torch import load_file
import torch

In [ ]:
vae = AutoEncoder(AutoEncoderParams())
transformer = Flux2(Klein4BParams())
vae_diction = load_file("/root/super_resolution/Edit-SR/checkpoints/FLUX.2-Klein-4B-Official/vae/diffusion_pytorch_model.safetensors")
transformer_diction = load_file("/root/super_resolution/Edit-SR/checkpoints/FLUX.2-Klein-4B-Official/transformer/diffusion_pytorch_model.safetensors")
vae_diction = checkpoint_transfer_diffusers2vae(vae_diction)
transformer_diction = checkpoint_transfer_diffusers2flux(transformer_diction)
vae.load_state_dict(vae_diction, strict=True)
transformer.load_state_dict(transformer_diction, strict=True)

In [ ]:
z_out = transformer(
    x=torch.zeros([1, 1024, 128]), # [1, 1024, 128]
    x_ids=torch.zeros([1, 1024, 4]), # [1, 1024, 4]
    timesteps=torch.tensor([0.], dtype=torch.float32), # [1,]
    ctx=torch.zeros([1, 256, 7680]), # [1, 256, 7680]
    ctx_ids=torch.zeros([1, 256, 4]), # [1, 256, 4]
    guidance=None,
)

In [ ]:
from src.models.qwen3vlemb import Qwen3VLEmbedder
import numpy as np
import torch

# Define a list of query texts
queries = [
    {"text": "A woman playing with her dog on a beach at sunset."},
    {"text": "Pet owner training dog outdoors near water."},
    {"text": "Woman surfing on waves during a sunny day."},
    {"text": "City skyline view from a high-rise building at night."}
]

# Define a list of document texts and images
documents = [
    {"text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust."},
    {"image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
    {"text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.", 
     "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"}
]

# Specify the model path
model_name_or_path = "/root/super_resolution/Edit-SR/checkpoints/Qwen3-VL-Embedding-2B"

# Initialize the Qwen3VLEmbedder model
model = Qwen3VLEmbedder(model_name_or_path=model_name_or_path)
# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# model = Qwen3VLEmbedder(model_name_or_path=model_name_or_path, torch_dtype=torch.float16, attn_implementation="flash_attention_2")

# Combine queries and documents into a single input list
inputs = queries + documents

# Process the inputs to get embeddings
embeddings = model.process(inputs)

# Compute similarity scores between query embeddings and document embeddings
similarity_scores = (embeddings[:4] @ embeddings[4:].T)

# Print out the similarity scores in a list format
print(similarity_scores.tolist())

# [[0.8157786130905151, 0.7178360223770142, 0.7173429131507874], [0.5195091962814331, 0.3302568793296814, 0.4391537308692932], [0.3884059488773346, 0.285782128572464, 0.33141762018203735], [0.1092604324221611, 0.03871120512485504, 0.06952016055583954]]


ModuleNotFoundError: No module named 'src.models.qwen3vlemb'

In [ ]:
similarity_scores.shape

torch.Size([4, 3])

In [ ]:
from src.models.pipeline import Pipeline
from PIL import Image

model = Pipeline(
    autoencoder_ckpt="/root/super_resolution/Edit-SR/checkpoints/FLUX.2-Klein-Base-4B/vae/diffusion_pytorch_model.safetensors",
    transformer_ckpt="/root/super_resolution/Edit-SR/checkpoints/FLUX.2-Klein-Base-4B/transformer/diffusion_pytorch_model.safetensors",
    text_encoder_ckpt="/root/super_resolution/Edit-SR/checkpoints/Qwen3-VL-Embedding-2B",
)

/root/miniconda3/envs/te/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 625/625 [00:00<00:00, 8765.10it/s]


In [ ]:
from src.quantize import quantize_modules
modules = quantize_modules([
    (model.transformer, "fp8"), 
    (model.text_encoder, "nvfp4"),
    (model.autoencoder, "bf16"),
])

In [ ]:
modules

[(Flux2(
    (pe_embedder): EmbedND()
    (img_in): Linear()
    (time_in): MLPEmbedder(
      (in_layer): Linear()
      (silu): SiLU()
      (out_layer): Linear()
    )
    (txt_in): Linear()
    (double_blocks): ModuleList(
      (0-4): 5 x DoubleStreamBlock(
        (img_norm1): LayerNorm()
        (img_attn): SelfAttention(
          (qkv): Linear()
          (norm): QKNorm(
            (query_norm): RMSNorm()
            (key_norm): RMSNorm()
          )
          (proj): Linear()
        )
        (img_norm2): LayerNorm()
        (img_mlp): Sequential(
          (0): Linear()
          (1): SiLUActivation(
            (gate_fn): SiLU()
          )
          (2): Linear()
        )
        (txt_norm1): LayerNorm()
        (txt_attn): SelfAttention(
          (qkv): Linear()
          (norm): QKNorm(
            (query_norm): RMSNorm()
            (key_norm): RMSNorm()
          )
          (proj): Linear()
        )
        (txt_norm2): LayerNorm()
        (txt_mlp): Sequential(


In [ ]:
img = Image.open("/root/super_resolution/Edit-SR/datasets/example.jpg")
w, h = img.size
new_w = max(16, round(w / 16) * 16)
new_h = max(16, round(h / 16) * 16)
img = img.resize((new_w, new_h), Image.BICUBIC)

x = model(
    images=[img], 
    gt_images=[img], 
    gt_texts=["1234567890"],
    training=True,
)

In [ ]:
from src.engine import Engine


In [ ]:
from transformer_engine.common.recipe import NVFP4BlockScaling, Float8BlockScaling, Format

nvfp4_recipe = NVFP4BlockScaling(
    fp4_format=Format.E2M1,
    disable_rht=False,
    disable_stochastic_rounding=False,
    disable_2d_quantization=False,
    interval=1,
    history_len=512,
    scaling_factor_whitelist=["weight", "activation", "gradient"]
)

In [ ]:
linear.quantizers

AttributeError: 'Linear' object has no attribute 'set_quantizer'

In [ ]:
import os
# 强制使用 triton 后端，这通常对 Float8Tensor 的算子支持最全
os.environ["TORCHAO_FP8_BACKEND"] = "triton" 

# 或者如果你是在特定的计算环境下
os.environ["TORCH_COMPILE_BACKEND"] = "inductor"

In [1]:
from src.engine import Engine, EngineConfig

/root/miniconda3/envs/imagesr/lib/python3.12/site-packages/torchao/dtypes/utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


In [2]:
engine = Engine(EngineConfig())

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

/root/miniconda3/envs/imagesr/lib/python3.12/site-packages/torchao/dtypes/utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
/root/miniconda3/envs/imagesr/lib/python3.12/site-packages/torchao/dtypes/uintx/semi_sparse_layout.py:84: UserWarning: Deprecation: SemiSparseLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
/root/miniconda3/envs/imagesr/lib/python3.12/site-packages/torchao/dtypes/uintx/tensor_core_tiled_layout.py:193: UserWarning: Deprecation: TensorCoreTiledLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


Using device: cuda


In [3]:
import transformer_engine.pytorch as te

engine.run_training()

NotImplementedError: Float8Tensor dispatch: attempting to run unimplemented operator/function: func=<OpOverload(op='aten.abs', overload='default')>, types=(<class 'torchao.quantization.Float8Tensor'>,), arg_types=(<class 'torchao.quantization.Float8Tensor'>,), kwarg_types={}

In [ ]:
len(engine.dataset)

0

In [ ]:
from pathlib import Path
list(Path("/root/super_resolution/Edit-SR/datasets/").glob("**/*.pdf"))

[PosixPath('/root/super_resolution/Edit-SR/datasets/example/00ae7b07fdece503a2deaa694c014024fe544629-5.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00a5e545397769fb34036209a1529690c6e43bb7-3.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00b20ebfa1d4b3c998e32373f959b37a7a9af87f-7.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00a79780cc73d5760cad810a8fa3b30b18b199c4-213.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00a05954f3cf7391cc2ebabeb66f703ce61ce302-22.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/000a71ba76308d5d84975b4541b1db30a6370c4e-53.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00b7e28459f61adb38e5ca14fe7ac1379817694b-29.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00a5f679776e545e0f407f581b9e4a646c298eaa-1.pdf'),
 PosixPath('/root/super_resolution/Edit-SR/datasets/example/00b6d2e98c4d77adff012167f22282730dae185d-2.pdf'),
 Posi